In [1]:
!pip install pymilvus==2.3.5
#!pip install openai==1.6.1
!pip install openai==0.28.1
!pip install langchain==0.0.354
!pip install tiktoken==0.5.2
!pip install transformers==4.36.2
!pip install pandas==2.1.4
!pip install pdfminer==20191125
!pip install pdfminer.six==20221105

# Connecting to Milvus

In [2]:
from pymilvus import connections

connections.add_connection(
    learn={
        "host": "localhost",
        "port": "19530",
        "username": "",
        "password": "",
    }
)

connection_id = "learn"
connections.connect(connection_id)
connections.list_connections()

D:\apps\anaconda3\envs\Milvus\Lib\site-packages\pymilvus\client\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


[('default', None),
 ('learn', <pymilvus.client.grpc_handler.GrpcHandler at 0x132ecbab710>)]

# Creating DB and Users

In [4]:
from pymilvus import db

current_dbs = db.list_database(using=connection_id)
print("Current databases:", current_dbs)

db_name="course_db"

if (db_name not in current_dbs):
    print("Creating Database:", db_name)
    wiki_db = db.create_database(db_name, using=connection_id)

db.using_database(db_name, using=connection_id)

Current databases: ['default', 'course_db']


# Creating Users

In [5]:
from pymilvus import Role, utility

current_users = utility.list_usernames(using=connection_id)
print("Current user list:", current_users)

Current user list: ['root']


# Creating Collections

In [20]:
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection
import json
#Define fields
course_id = FieldSchema (
    name="course_id",
    dtype=DataType. INT64,
    is_primary =True,
    max_length=32)
title = FieldSchema (
    name="title",
    dtype=DataType. VARCHAR,
    max_length=256)
description = FieldSchema (
    name="description",
    dtype=DataType. VARCHAR,
    max_length=2048)

#Dim should match the embedding size
desc_embedding = FieldSchema (
    name="desc_embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=1536
)

#Define Schema
wiki_schema=CollectionSchema(
    fields=[course_id, title, description, desc_embedding],
    description="Course List",
    enable_dynamic_field=True
)

col_name="courses_list"

wiki_collection = Collection(
    name=col_name,
    schema=wiki_schema,
    using=connection_id,
    shard_num=2
)

from pymilvus import utility

print("Current collections: ",utility.list_collections(using=connection_id))

#setup existing collection into another object
r_collection = Collection(col_name, using=connection_id)
print("\n", r_collection.schema)

Current collections:  ['courses_list']

 {'auto_id': False, 'description': 'Course List', 'fields': [{'name': 'course_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048}}, {'name': 'desc_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}], 'enable_dynamic_field': True}


# Inserting Data in Milvus

In [6]:
import pandas as pd
course_descriptions = pd.read_csv("sample.csv")
course_descriptions.head()

,Course ID,Title,Description
0,1001,clean-IT: Towards Sustainable Digital Technolo...,Digitalization is a game changer in the pursui...
1,1002,HPI Academy: Strategic Design Thinking For Eve...,This is an online course for everyone who want...
2,1003,Algorithmen und Datenstrukturen mit Detektiv Duke,Du hast bereits den Java-Kurs auf openHPI gema...
3,1004,Digital Future Literacy - Fit für die digitale...,Im Zuge der Digitalisierung müssen sich viele ...
4,1005,Design Thinking 4.0 – The Cultural Dimension o...,"Design Thinking is an innovation approach, whi..."


In [11]:
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
import os

openai_api_key = "abc123"
os.environ["OPENAI_API_KEY"] = openai_api_key

embeddings_model = OpenAIEmbeddings()

In [15]:
#Preparing Data
i_course_id = course_descriptions["Course ID"].tolist()
i_title = course_descriptions["Title"].tolist()
i_description = course_descriptions["Description"].tolist()

i_desc_embeddings = [embeddings_model.embed_query(i) for i in i_description]

insert_data = [i_course_id, i_title, i_description, i_desc_embeddings]

In [21]:
#Init Collection object
course_collection = Collection(col_name, using=connection_id)
#Insert Data
mr = course_collection.insert(insert_data)

#Flush the data after insert
print("Inserted data. Now flushing")
course_collection.flush(timeout=180)

Inserted data. Now flushing


# Building Index

In [23]:
index_params = {
    "metric_type": "L2",
    "index_type": "IVF_FLAT",
    "params": { "nlist": 1024 },
}

course_collection.create_index(
    field_name="desc_embedding",
    index_params=index_params
)
utility.index_building_progress(col_name, using=connection_id)

{'total_rows': 100, 'indexed_rows': 100, 'pending_index_rows': 0}

# Query

In [24]:
course_collection.load()
print("Course Collection Loaded")

Course Collection Loaded


In [25]:
q_result = course_collection.query(
    expr = "course_id == 1001",
    output_fields = ["title", "description"]
)
print(q_result)
print("\n Result object :", type(q_result[0]))

[{'title': 'clean-IT: Towards Sustainable Digital Technologies', 'description': 'Digitalization is a game changer in the pursuit of a sustainable future. The latest digital technologies and applications like cloud, AI, and mobile devices enable us to achieve the Sustainable Development Goals and reduce carbon emissions in many sectors. Yet computer systems themselves have an immense energy requirement for their countless devices, data centers, applications and global networks. To effectively reduce the carbon footprint of digitalization, it is necessary to apply algorithmic efficiency and sustainability by design as guiding principles in digital engineering. The clean-IT Forum is the international platform to exchange ideas, recent research findings and applications to make digital technologies more energy-efficient.', 'course_id': 1001}]

 Result object : <class 'dict'>


In [26]:
q_result2 = course_collection.query(
    expr = "(title LIKE 'Cyber%') && (course_id > 1001)",
    output_fields = ["title", "description"]
)
print(q_result2)

[{'title': 'Cybersecurity Basics - Exam', 'description': 'If you have completed the three security workshops offered on openHPI this year, you should now have a good knowledge of the basics of cybersecurity. Here we offer you the opportunity to take an exam on the material from all three workshops. If successful, you will receive a record of achievement or a qualified certificate, which is usually only awarded in 6-week courses.\nYou may not register for the exam until you have met the eligibility requirements. The following criteria must be met:\nParticipation in the 3 Workshops Confidential Communication in the Internet, Digital Identities, Cyberthreats by Malware\nProof of a confirmation of participation in each of the three workshops (access of at least 80% of the learning material)\nThe participation in the weekly assignments in the individual courses is not relevant for this exam.', 'course_id': 1020}, {'title': 'Cyberthreats by Malware', 'description': 'In this course we will di

# Vector Search Query

In [27]:
search_params = {
    "metric_type": "L2",
    "offset": 0,
    "ignore_growing": False,
    "params": { "nprobe": 10 }
}
# Vector Embed of Input String
search_string = "cyber"
search_embed = embeddings_model.embed_query(search_string)

#Perform Search
s_results = course_collection.search(
    data=[search_embed],
    anns_field="desc_embedding",
    param=search_params,
    limit=10,
    expr=None,
    output_fields=["title"],
    consistency_level="Strong"
)
print("Search result object:", type(s_results[0]), "\n")

for i in s_results[0]:
    print(i.id, str(round(i.distance, 2)), "\t", i.entity.get("title"))

Search result object: <class 'pymilvus.client.abstract.Hits'> 

1063 0.37 	 Internet Security for Beginners
1080 0.38 	 Web Technologies
1022 0.39 	 Cyberthreats by Malware
1079 0.39 	 Sicherheit im Internet
1035 0.39 	 Tatort Internet - Angriffsvektoren und Schutzmaßnahmen
1020 0.39 	 Cybersecurity Basics - Exam
1066 0.4 	 Social Media - What No One has Told You about Privacy
1091 0.4 	 Social Media - What No One has Told You about Privacy
1001 0.4 	 clean-IT: Towards Sustainable Digital Technologies
1024 0.41 	 Confidential Communication in the Internet


In [28]:
# Search an unrelated query
search_string2 = "best movies of the year"
search_embed2 = embeddings_model.embed_query(search_string2)

#Perform Search
s_results2 = course_collection.search(
    data=[search_embed2],
    anns_field="desc_embedding",
    param=search_params,
    limit=10,
    expr=None,
    output_fields=["title"],
    consistency_level="Strong"
)
print("Search result object:", type(s_results[0]), "\n")

for i in s_results2[0]:
    print(i.id, str(round(i.distance, 2)), "\t", i.entity.get("title"))

Search result object: <class 'pymilvus.client.abstract.Hits'> 

1080 0.58 	 Web Technologies
1058 0.58 	 Java Capstone Series Pt. 1
1015 0.59 	 Learning @ Scale 2021 & EMOOCs 2021
1025 0.59 	 Future of Computing - on the Road to Quantum
1068 0.6 	 In-Memory Data Management
1032 0.61 	 HPI Academy: Leading Digital Transformation and Innovation - Fall 2020
1022 0.61 	 Cyberthreats by Malware
1036 0.61 	 Co-Innovation Journey for Startups and Corporates
1048 0.62 	 Future of Computing - IBM POWER9 and beyond
1001 0.62 	 clean-IT: Towards Sustainable Digital Technologies


# Deleting Object

In [29]:
# Deleting SIngle Record
course_collection.delete("course_id in [1002]")

(insert count: 0, delete count: 1, upsert count: 0, timestamp: 464416447501434881, success count: 0, err count: 0)